# WEEK 2: SAR基礎 — 画像から入る 演習ノート

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astrocamp-2026-siaPpts/astrocamp-sat-gis/blob/main/public/notebooks/exercises/week2_exercise.ipynb)

> **Private リポジトリの注意:**
> このリポジトリは Private 設定のため、上のバッジから直接開けない場合があります。
> その場合は以下の手順で開いてください：
>
> 1. [Google Colab](https://colab.research.google.com/) にアクセス
> 2. 「ファイル」→「ノートブックを開く」→「GitHub」タブ
> 3. 「プライベート リポジトリを含める」にチェックを入れる
> 4. 初回はGitHub認証が求められるので許可する
> 5. リポジトリ `astrocamp-2026-siaPpts/astrocamp-2026-sia` を選択し、該当ノートブックを開く

このノートブックでは Google Earth Engine (GEE) を使って **Sentinel-1 SAR** 画像を扱い、
光学画像との違い・散乱機構・偏波の違いを体験する。

---


In [ ]:
# 必要なライブラリのインストール（Colab 環境の場合）
!pip install geemap earthengine-api

In [ ]:
import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
from getpass import getpass

# ── 実行環境の判定 ──
_IN_COLAB = 'google.colab' in sys.modules

# ── GEE の初期化（3段階フォールバック） ──
def _init_gee():
    """Try default → env var → manual input, never raises."""
    # Stage 1: try default credentials
    try:
        ee.Initialize()
        project = ee.data.getAssetRoots()[0]['name'].split('/')[-1]
        print(f'✅ GEE initialized (default project: {project})')
        return
    except Exception:
        pass

    # Stage 2: environment variable
    project = os.environ.get('GEE_PROJECT')
    if project:
        try:
            ee.Initialize(project=project)
            print(f'✅ GEE initialized with project: {project}')
            return
        except Exception as e:
            print(f'⚠  GEE_PROJECT={project} での初期化に失敗: {e}')

    # Stage 3: manual input
    print('=' * 60)
    print('GEE の認証が必要です。')
    print('手順:')
    print('  1. https://signup.earthengine.google.com/ でGEEに登録')
    print('  2. https://code.earthengine.google.com/ にアクセスし、プロジェクトIDを確認')
    print('  3. ターミナルで以下を実行:')
    print('     python -c "import ee; ee.Authenticate()"')
    print('=' * 60)
    project = input('GEE project ID を入力 (例: ee-yourproject): ').strip()
    if project:
        ee.Initialize(project=project)
        print(f'✅ GEE initialized with project: {project}')
    else:
        print('⚠ プロジェクトIDが入力されませんでした。後続のセルは失敗します。')

if _IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    ee.Authenticate()

_init_gee()


def _auto_clip(arr, low=2, high=98):
    valid = arr[~np.isnan(arr)]
    if len(valid) == 0:
        return -25, 0
    return float(np.percentile(valid, low)), float(np.percentile(valid, high))


# ── matplotlib 日本語表示 ──
plt.rcParams['font.family'] = 'AppleGothic'

In [ ]:
# ===== 座標設定 =====
# 座標を変更する場合はこのセルだけを編集してください
ROI = ee.Geometry.Rectangle([140.5, 37.2, 141.2, 37.8])
DISPLAY_ROI = ee.Geometry.Rectangle([140.90, 37.77, 141.01, 37.83])

# SAR散乱機構確認地点（相馬市）
WATER_PT     = ee.Geometry.Point([140.9780, 37.8000])
URBAN_PT     = ee.Geometry.Point([140.9278, 37.7925])
FARMLAND_PT  = ee.Geometry.Point([140.9367, 37.7948])
FOREST_PT    = ee.Geometry.Point([140.9118, 37.7944])
RIVERBED_PT  = ee.Geometry.Point([140.9385, 37.8008])
FARM_POINT   = ee.Geometry.Point([140.9644, 37.8041])

# ====================

### 学習目標

この演習が終わった後にできること：

1. SAR画像の明るさを **3つの散乱機構（表面・体積・ダブルバウンス）** に紐づけて説明できる
2. **VV偏波とVH偏波** の違いと、それぞれが敏感な地物の違いを説明できる
3. **光学画像では区別できないがSARでは区別できる** 地物の例を挙げられる
4. 生成AIのSAR解説に潜む物理的誤りを見抜ける

### 前提知識

SARの物理原理（能動センサ・波長と散乱・3つの散乱機構・偏波）については、講義資料（CHAPTER 01-02）を参照すること。

### VSCode で実行する場合

1. リポジトリルートで `uv sync` を実行
2. VSCode カーネル選択 → `.venv/bin/python`
3. 事前に `python -c "import ee; ee.Authenticate()"` でGEE認証
4. 環境変数 `GEE_PROJECT` を設定しておくとプロンプトがスキップできる

---


### SARの基礎知識 ― 合成開口と周波数バンド

#### なぜ「合成開口レーダー（SAR）」と呼ばれるのか

SAR（Synthetic Aperture Radar）の「合成開口」の意味を理解するには、前回（W1）のアンテナ理論に立ち返る。
アンテナの角度分解能は $\theta \approx \lambda / D$ で決まる。
Sentinel-1のC-bandマイクロ波（$\lambda \approx 5.6\,\text{cm}$）で高度700kmから地上分解能10mを得ようとすると、
**実開口では直径約4kmのアンテナが必要**になる。これは物理的に不可能だ。

SARはこの問題を**衛星自身の運動**で解決する。
衛星が約7.5km/sで移動しながら同じ地点に連続的にパルスを照射し、
その軌跡上の各位置で受信した信号の位相・振幅を記録して数学的に合成することで、
「長さ約2kmの仮想アンテナ」を実現する。
これが「合成開口（Synthetic Aperture）」の所以であり、SARという名称の由来である。

#### SARの周波数バンド

衛星SARには使用するマイクロ波の波長によっていくつかの「バンド」がある：

| バンド | 波長 | 代表的な衛星 | 特徴 |
|-------|------|-------------|------|
| **X-band** | ~3cm | TerraSAR-X, COSMO-SkyMed | 高分解能（1m級）、都市域の詳細監視に強い |
| **C-band** | ~5.6cm | **Sentinel-1**, RADARSAT | 分解能と透過性のバランス。農業監視に最適 |
| **L-band** | ~24cm | **ALOS-2**（だいち2号） | 植生透過性が高く、森林・バイオマスに有効 |

本演習ではSentinel-1のC-bandデータを使用する。
**発展：** L-bandのALOS-2データはJAXA Earth APIからアクセス可能。興味があれば後日比較してみよう。

---


In [ ]:
# ── 解析対象の範囲 ──
# 福島県相馬市周辺（W1と同じ範囲）
# 表示用（matplotlib 描画用、全体ROIと同じ）


---
## STEP 1: 5つのSAR画像から1枚を選ぶ

配布された5つの観測条件から **1つ選び**、その画像を取得しましょう。
以下の5つは、時期・軌道方向・偏波が異なります。

| # | 観測時期 | 軌道方向 | 特徴 |
|---|---------|---------|------|
| ① | 2024-05-15 | DESCENDING | 代かき・湛水期 — 水田が最も暗くなる時期 |
| ② | 2024-07-10 | DESCENDING | 生育期ピーク — イネが最も茂る時期 |
| ③ | 2024-10-05 | DESCENDING | 収穫後 — 裸地に戻る時期 |
| ④ | 2024-07-15 | DESCENDING | ②と同じ7月だが軌道方向違い |
| ⑤ | 2025-01-10 | DESCENDING | 冬季 — 雪の有無・落葉後の様子 |

> **選び方のヒント：** 「どの季節のSAR画像を見ると光学との違いが一番はっきりするか？」を考えて選んでみよう。


In [ ]:
# ★ EETODO: 下のリストから見たい条件の番号 (1〜5) を選んで date_str と orbit を設定せよ
options = {
    1: {'date': ('2024-05-01', '2024-05-31'), 'orbit': 'DESCENDING', 'label': '① 湛水期 (2024-05)'},
    2: {'date': ('2024-07-01', '2024-07-31'), 'orbit': 'DESCENDING', 'label': '② 生育期 (2024-07)'},
    3: {'date': ('2024-10-01', '2024-10-31'), 'orbit': 'DESCENDING', 'label': '③ 収穫後 (2024-10)'},
    4: {'date': ('2024-07-01', '2024-07-31'), 'orbit': 'DESCENDING', 'label': '④ 生育期 (2024-07) DESC'},
    5: {'date': ('2025-01-01', '2025-01-31'), 'orbit': 'DESCENDING', 'label': '⑤ 冬季 (2025-01)'},
}

choice = 4  # ← ここを 1〜5 で変更
opt = options[choice]

# Sentinel-1 GRD コレクションから検索
s1_collection = (
    ee.ImageCollection(???)
    .filterBounds(ROI)
    .filterDate(opt['date'][0], opt['date'][1])
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', ???))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.eq('orbitProperties_pass', opt['orbit']))
)

count = s1_collection.size().getInfo()
print(f'条件に一致するシーン数: {count}')

image = None
if count > 0:
    image = s1_collection.first()
    # ── メタデータの読み取り ──
    date_obs = image.date().format('YYYY-MM-dd').getInfo()
    orbit_dir = image.get('orbitProperties_pass').getInfo()
    pol = image.get('transmitterReceiverPolarisation').getInfo()
    rel_orbit = image.get('relativeOrbitNumber_start').getInfo()
    print(f'📅 観測日: {date_obs}')
    print(f'🛰 軌道方向: {orbit_dir}')
    print(f'📡 偏波: {pol}')
    print(f'🔢 相対軌道番号: {rel_orbit}')
    print(f'✅ 選択: {opt["label"]}')
# 考察（必須）:
# Q1. 選択した条件の観測日・軌道方向・偏波を記録せよ。
# Q2. ASCENDINGとDESCENDINGで、見つかるシーン数に差はあるか？ その理由を調べよ。
# Q3. 実際に取得された画像の観測日と、optionsで指定した月がずれることがある。なぜか？
# → ここに考察を書く


> **ヒント STEP 1**
>
> - 各条件に対して実際に見つかるシーン数は1枚とは限らない。フィルタ結果が複数の場合は `first()` で最初の1枚を取得している。
> - 選択した画像のメタデータ（観測日・軌道方向・偏波）は、**Exercise Log の課題①に必ず記録する**。
> - もし `count == 0` になった場合は、他の番号を試してみよう。
>
> **深堀り:** `relativeOrbitNumber_start` が何を示すか調べてみよう。


---
## STEP 2: SAR画像を表示する（VV vs VH）

選んだ画像の VV偏波・VH偏波を並べて表示します。
SAR画像の「明るさ＝後方散乱係数 σ⁰ [dB]」で表現します。


In [ ]:
if 'image' not in dir() or image is None:
    print('❌ STEP 1 が未実行です。「STEP 1 → STEP 2 → STEP 3」の順に実行してください。')
    print('ヒント: choice の値を 1〜5 の間で変えてみよう。')
else:
    # ── VV・VH は既に σ⁰ [dB] 値
    vv_db = image.select(???).rename('VV_db')
    vh_db = image.select(???).rename('VH_db')

    print('VV画像を取得中... (scale=100)')
    vv_arr = geemap.ee_to_numpy(vv_db, region=DISPLAY_ROI, scale=100)
    print('VH画像を取得中... (scale=100)')
    vh_arr = geemap.ee_to_numpy(vh_db, region=DISPLAY_ROI, scale=100)
    # ── 並べて表示 ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    vmin0, vmax0 = _auto_clip(vv_arr)
    im0 = axes[0].imshow(vv_arr, cmap='gray', vmin=vmin0, vmax=vmax0)
    axes[0].set_title(f'VV (Co-pol) - {opt["label"]}', fontsize=12)
    axes[0].axis('off')
    plt.colorbar(im0, ax=axes[0], shrink=0.7, label=???)

    vmin1, vmax1 = _auto_clip(vh_arr)
    im1 = axes[1].imshow(vh_arr, cmap='gray', vmin=vmin1, vmax=vmax1)
    axes[1].set_title(f'VH (Cross-pol) - {opt["label"]}', fontsize=12)
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1], shrink=0.7, label=???)

    plt.tight_layout()
    plt.show()


# 考察（必須）:
# Q1. VV偏波とVH偏波で、水域・都市・森林の明るさはどう違うか？
# Q2. なぜ水域はどちらの偏波でも暗いのか、散乱機構の観点から説明せよ。
# Q3. 都市部がVVで明るい理由を、ダブルバウンスと関連付けて説明せよ。
# → ここに考察を書く


> **ヒント STEP 2**
>
> - 表示範囲が海岸部を含む場合、海面は非常に暗い（-20dB以下）ので、画面上で黒く見える。
> - VVとVHの **差** に注目：同じ場所でも2つの画像で明るさが異なるはず。
> - スライダーや拡大はできないが、ざっくりとした傾向を把握しよう。
>
> **Exercise Log 課題①の記録：** 選んだ画像の中で「最も明るい場所」と「最も暗い場所」はどこか？ それぞれの散乱機構を答えよう。


---
## STEP 3: 光学画像（Sentinel-2）と比較する

同じエリアの Sentinel-2 光学画像を取得し、SAR画像と比べてみましょう。
「光学では区別できないが、SARでは区別できる」場所を探すのがゴールです。


In [ ]:
# STEP 2 が未実行の場合はガード
if 'vv_arr' not in dir() or vv_arr is None:
    print('❌ STEP 2 が未実行です。「STEP 1 → STEP 2 → STEP 3」の順に再実行してください。')
else:
    # 同じ時期の光学画像（なるべく雲の少ないもの）を取得
    s2_collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(ROI)
        .filterDate(opt['date'][0], opt['date'][1])
        .filterMetadata('CLOUDY_PIXEL_PERCENTAGE', 'less_than', 30)
    )

    if s2_collection.size().getInfo() == 0:
        print('同じ月に雲30%以下のS-2画像なし → 期間を前後2週間に拡大')
        from datetime import datetime, timedelta
        mid = datetime.strptime(opt['date'][0], '%Y-%m-%d') + timedelta(days=15)
        s2_collection = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(ROI)
            .filterDate(
                (mid - timedelta(days=14)).strftime('%Y-%m-%d'),
                (mid + timedelta(days=14)).strftime('%Y-%m-%d')
            )
            .sort('CLOUDY_PIXEL_PERCENTAGE')
        )

    if s2_collection.size().getInfo() == 0:
        print('❌ 拡大してもS-2画像が見つかりませんでした。光学比較はスキップします。')
    else:
        s2_image = s2_collection.first()
        s2_date = s2_image.date().format('YYYY-MM-dd').getInfo()
        s2_cloud = s2_image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()

        print(f'S-2 観測日: {s2_date} (雲量: {s2_cloud:.0f}%)')
        print('光学画像を取得中... (scale=100)')
        rgb_arr = geemap.ee_to_numpy(s2_image.select([???, ???, ???]), region=DISPLAY_ROI, scale=100)

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        _vmin, _vmax = _auto_clip(vv_arr)
        axes[0].imshow(vv_arr, cmap='gray', vmin=_vmin, vmax=_vmax)
        axes[0].set_title('SAR VV [dB]', fontsize=12)
        axes[0].axis('off')

        axes[1].imshow(vh_arr, cmap='gray', vmin=_vmin, vmax=_vmax)
        axes[1].set_title('SAR VH [dB]', fontsize=12)
        axes[1].axis('off')

        rgb_display = np.clip((rgb_arr - 0) / (??? - 0), 0, 1)
        axes[2].imshow(rgb_display)
        axes[2].set_title(f'Optical RGB ({s2_date})', fontsize=12)
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()
# 考察（必須）:
# Q1. SAR画像（VV/VH）と光学画像（RGB）で、農地の見え方はどう異なるか？
# Q2. 「光学では区別できないがSARでは区別できる」場所を1つ挙げ、その理由を説明せよ。
# Q3. 光学画像に雲がある場所は、SAR画像ではどう写っているか？
# → ここに考察を書く


> **STEP 3 の観察ポイント**
>
> 3つの画像を見比べて、以下の点を Exercise Log に記録しよう。
>
> 1. **光学で似ているがSARで異なる：** 収穫後の農地と休耕地は光学RGBでは両方「茶色い地面」に見える。SARではどうか？
> 2. **光学で異なるがSARで似ている：** 例えば、植生の種類の違いは光学では色で区別できるが、SARではどうか？
> 3. **雲の有無：** もしS-2画像に雲がかかっている場所があれば、SARではその下も見えているか確認しよう。
>
> **Exercise Log 課題①に必須：** 「光学では区別できないがSARでは異なる明るさになる」エリアを **1箇所以上** 特定し、その理由を散乱機構から説明せよ。


---
## STEP 4: 散乱機構を定量的に確認する

画像上の代表的な地点で後方散乱係数 σ⁰ の値を読み取り、
3つの散乱機構（表面散乱・体積散乱・ダブルバウンス）に分類します。


In [ ]:
if 'vv_db' not in dir() or vv_db is None:
    print('❌ STEP 2 が未実行です。「STEP 1 → STEP 2 → STEP 4」の順に再実行してください。')
else:
    # 各地物タイプの代表点
    points = {
    '水域 (Water/ocean)': WATER_PT,
    '都市 (Urban/Soma)': URBAN_PT,
    '農地 (Farmland)': FARMLAND_PT,
    '山林 (Forest)': FOREST_PT,
    '河川敷 (Riverbed)': RIVERBED_PT,
}

    print(f'{"地物タイプ":<20} {"VV σ⁰ [dB]":<15} {"VH σ⁰ [dB]":<15} {"VV-VH差":<15} {"推定される散乱機構":<25}')
    print('-' * 90)

    results = []
    for name, pt in points.items():
        vv_val = vv_db.sample(pt, 10).first()
        vh_val = vh_db.sample(pt, 10).first()
        if vv_val and vh_val:
            vv_mean = vv_val.get('VV_db').getInfo()
            vh_mean = vh_val.get('VH_db').getInfo()
            diff = vv_mean - vh_mean
            results.append({'name': name, 'vv': vv_mean, 'vh': vh_mean, 'diff': diff})
            print(f'{name:<20} {vv_mean:<15.1f} {vh_mean:<15.1f} {diff:<15.1f} {"(要分類)":<25}')
        else:
            print(f'{name:<20} {"N/A":<15} {"N/A":<15} {"N/A":<15} {"—":<25}')

    print()
    print('★ EETODO: 上の表の「推定される散乱機構」の列を、以下の選択肢から埋めよ。')
    print('   選択肢: 表面散乱(鏡面) / 表面散乱(粗面) / 体積散乱 / ダブルバウンス')
    print('   ヒント: σ⁰が-20dB以下→表面散乱(鏡面), -15〜-8dB→体積散乱, -5dB以上→ダブルバウンス')
# ★ EETODO: 上の表の結果を参考に、各地点の散乱機構をコード上で記入せよ
#   選択肢: '表面散乱(鏡面)', '表面散乱(粗面)', '体積散乱', 'ダブルバウンス'
scattering_labels = {
    '水域 (Water/ocean)': ???,
    '都市 (Urban/Soma)': ???,
    '農地 (Farmland)': ???,
    '山林 (Forest)': ???,
    '河川敷 (Riverbed)': ???,
}
# 考察（必須）:
# Q1. 水域の散乱機構を表面散乱(鏡面)と分類した理由を、σ⁰の値に基づいて説明せよ。
# Q2. 都市と森林でVV-VH差が異なるのはなぜか？
# Q3. 農地の散乱機構は何か？ 季節によって変化する理由を考察せよ。
# → ここに考察を書く


> **STEP 4 の記録**
>
> 上の結果を Exercise Log に転記し、**各地点の散乱機構を1つ選んで記入** せよ。
>
> | 地物 | VV σ⁰ | VH σ⁰ | 散乱機構 | 選んだ理由 |
> |---|---|---|---|---|
> | 水域 | (値) | (値) | 表面散乱(鏡面) | 水面は滑らかなので電波が反射して飛んでいく |
> | 都市 | ... | ... | ... | ... |
>
> **発展（B群向け）:** VV-VH差が大きい/小さい理由を物理的に説明してみよう。


---
## STEP 5: 生成AIにSAR画像を説明させてみる（必須課題②）

生成AI（ChatGPT/Gemini）にSAR画像を読み込ませて説明を生成させ、
**物理的に誤っている記述を1つ以上見つけて記録** する。

### プロンプト例（コピーしてAIに入力）

```
以下の画像はSentinel-1 SAR（Cバンド、2024年7月、福島県相馬市）の画像です。
VV偏波とVH偏波のグレースケール画像です。

この画像に写っている地物を説明してください。特に以下の点に触れてください：
1. 中央付近の暗い領域は何か
2. 明るく光っている領域は何か
3. 光学画像と比べて、SARでしか見えない特徴は何か
```

> **注意:** AIに画像を「理解させる」方法はツールによって異なる。
> - ChatGPT Plus (GPT-4系): 画像をアップロードできる
> - Gemini: 画像を添付できる
> - Claude: 画像を添付できる
>
> いずれの場合も、**AIの出力をそのまま信じず、Step 1〜4の知識を使ってファクトチェックすること**。


---
## ⭐ チャレンジ課題（発展・任意）

以下の課題に **最低1問以上** 挑戦し、結果を Exercise Log に記録しよう。


---
### ⭐ C1: 時期の違いによるSAR画像の比較

**問題:** 同じDESCENDING軌道で、5月（湛水期）と7月（生育期）のSAR画像を両方取得し、
時期による後方散乱の変化を比較せよ。

**観察ポイント:**
- 5月（代かき後）と7月（生育期）で明るさはどう変わるか？
- 水域・農地・都市それぞれの変化を観察しよう。


In [ ]:
# ★ EETODO: 5月（湛水期）と7月（生育期）のSAR画像を比較
options_c1 = {
    '5月(湛水期)':  {'date': ('2024-05-01', '2024-05-31'), 'orbit': 'DESCENDING'},
    '7月(生育期)': {'date': ('2024-07-01', '2024-07-31'), 'orbit': 'DESCENDING'},
}

results_c1 = {}
for label, cfg in options_c1.items():
    col = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(ROI)
        .filterDate(cfg['date'][0], cfg['date'][1])
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.eq('orbitProperties_pass', cfg['orbit']))
    )
    if col.size().getInfo() > 0:
        img = col.first()
        vv = img.select('VV')
        print(f'  {label}: scale=100でラスタ取得中...')
        arr = geemap.ee_to_numpy(vv, region=DISPLAY_ROI, scale=100)
        results_c1[label] = arr
        print(f'{label}: {img.date().format("YYYY-MM-dd").getInfo()} 取得完了')
    else:
        print(f'{label}: 該当シーンなし')

if len(results_c1) == 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, (label, arr) in zip(axes, results_c1.items()):
        _vmin, _vmax = _auto_clip(arr)
        im = ax.imshow(arr, cmap='gray', vmin=_vmin, vmax=_vmax)
        ax.set_title(f'VV - {label}', fontsize=12)
        ax.axis('off')
        plt.colorbar(im, ax=ax, shrink=0.7, label='σ⁰ [dB]')
    plt.tight_layout()
    plt.show()
    print('⭐ 考察: 2つの画像で明るさや陰影の付き方がどう違うか観察しよう。')
    print('    特に建物や林縁の見え方に注目。')# 考察（必須）:
# Q1. 5月と7月で、農地の明るさはどう変化したか？
# Q2. 代かき（湛水）によって水田のσ⁰が低下する理由を説明せよ。
# Q3. 水域（海・湖）は2つの画像で変化しているか？ その理由は？
# → ここに考察を書く


---
### ⭐ C2: VV/VH差マップで見る偏波の情報

**問題:** VVとVHの差 (VV - VH) のマップを作成し、偏波間差が大きい場所・小さい場所を可視化せよ。
これにより、作付け水田と休耕地の違いがより明確になる。


In [ ]:
if 'vv_db' not in dir() or vv_db is None:
    print('❌ STEP 2 が未実行です。「STEP 1 → STEP 2」を先に実行してから再実行してください。')
else:
    # VV - VH 差マップの計算
    diff_db = ???.subtract(???).rename('VV_minus_VH')

    print('VV-VH差マップを取得中...')
    print('VV-VH差マップを取得中... (scale=100)')
    diff_arr = geemap.ee_to_numpy(diff_db, region=DISPLAY_ROI, scale=100)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    _vmin0, _vmax0 = _auto_clip(vv_arr)
    im0 = axes[0].imshow(vv_arr, cmap='gray', vmin=_vmin0, vmax=_vmax0)
    axes[0].set_title('VV [dB]', fontsize=12)
    axes[0].axis('off')

    _vmin1, _vmax1 = _auto_clip(vh_arr)
    im1 = axes[1].imshow(vh_arr, cmap='gray', vmin=_vmin1, vmax=_vmax1)
    axes[1].set_title('VH [dB]', fontsize=12)
    axes[1].axis('off')

    im2 = axes[2].imshow(diff_arr, cmap='RdBu_r', vmin=0, vmax=15)
    axes[2].set_title('VV - VH 差 [dB]', fontsize=12)
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], shrink=0.7, label='VV-VH [dB]')

    plt.tight_layout()
    plt.show()

    print('⭐ 考察:')
    print('  - 差が大きい（赤っぽい）場所 → 都市・建物（ダブルバウンスでVVのみ強いため差が開く）')
    print('  - 差が小さい（青っぽい）場所 → 森林・草地（体積散乱で両偏波が同程度）や水面（両偏波ともノイズフロア付近）')
    print('    ※ 平滑な水面はVVもVHも非常に暗く（ノイズフロア）、差の値自体が安定しない')
    print('  - 農地に注目: 作付け中と収穫後で差はどう変わるか？')# 考察（必須）:
# Q1. 差が大きい場所（赤っぽい）はどのような地物か？ その散乱機構は？
# Q2. 差が小さい場所（青っぽい）はどのような地物か？
# Q3. VV-VH差マップは、作付け水田と休耕地の区別に有効か？
# → ここに考察を書く


---
### ⭐ C3: 時系列で見る作付けサイクル

**問題:** 同じ農地上の1点について、異なる月のSAR画像を取得し、
σ⁰の季節変化をグラフにプロットせよ。

代かき（5月）の急激なσ⁰低下と、生育期（7-8月）の上昇が捉えられるか確認しよう。


In [ ]:
# ★ EETODO: 農地上の1点を設定せよ（デフォルトは相馬市）

# 2024年の各月のSAR画像を取得
months = [
    ('2024-04', '2024-04-01', '2024-04-30'),
    ('2024-05', '2024-05-01', '2024-05-31'),
    ('2024-06', '2024-06-01', '2024-06-30'),
    ('2024-07', '2024-07-01', '2024-07-31'),
    ('2024-08', '2024-08-01', '2024-08-31'),
    ('2024-09', '2024-09-01', '2024-09-30'),
    ('2024-10', '2024-10-01', '2024-10-31'),
]

ts_data = []
for label, start, end in months:
    col = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(FARM_POINT)
        .filterDate(start, end)
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
    )
    count = col.size().getInfo()
    if count > 0:
        img = col.first()
        vv_sample = img.select('VV').sample(FARM_POINT, 100)
        vh_sample = img.select('VH').sample(FARM_POINT, 100)
        if vv_sample.size().getInfo() > 0 and vh_sample.size().getInfo() > 0:
            vv_mean = vv_sample.first().get('VV').getInfo()
            vh_mean = vh_sample.first().get('VH').getInfo()
            ts_data.append({
                'month': label,
                'vv': vv_mean,
                'vh': vh_mean
            })
            print(f'{label}: VV={vv_mean:.1f} dB, VH={vh_mean:.1f} dB')
    else:
        print(f'{label}: データなし')

if len(ts_data) >= 3:
    months_label = [d['month'] for d in ts_data]
    vv_vals = [d['vv'] for d in ts_data]
    vh_vals = [d['vh'] for d in ts_data]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(months_label, vv_vals, 'o-', color='red', label='VV')
    ax.plot(months_label, vh_vals, 's--', color='blue', label='VH')
    ax.set_xlabel('Month')
    ax.set_ylabel('σ⁰ [dB]')
    ax.set_title('農地1点のSAR後方散乱係数の月変化')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    print('⭐ 考察:')
    print('  - 5月に σ⁰ の急落は見られるか？ → 湛水（代かき）の有無')
    print('  - 7-8月に上昇は見られるか？ → イネの生長')
    print('  - VVとVHの差は季節によってどう変化するか？')
else:
    print('データが不足しています。FARM_POINT の座標を農地区画に変更してみてください。')
# 考察（必須）:
# Q1. 5月にσ⁰の急落は見られるか？ それはなぜか？
# Q2. VVとVHの差は季節によってどう変化するか？ その理由を考察せよ。
# Q3. もし休耕地のデータがあれば、作付け中の農地とどう違うと考えられるか？
# → ここに考察を書く


---
## まとめ

### 今日確認したこと

- [x] SAR画像の明るさ ＝ 後方散乱係数 σ⁰ [dB]
- [x] 3つの散乱機構：表面散乱（暗い）・体積散乱（グレー）・ダブルバウンス（明るい）
- [x] 偏波の違い：VV（表面散乱に敏感） vs VH（体積散乱に敏感）
- [x] 光学とSARの違い：雲の有無・地物の区別の仕方の違い
- [x] AIのSAR解説には物理的誤りが含まれる可能性がある

### 次週への接続

- **Week 3（SAR実践）:** 今回「見る」だけだったSARを、自分でデータを選び、入射角や観測モードを考慮しながら解析する。
- **Week 5（変化検出＋セグメンテーション）:** 今回の「時系列パターン」の考え方をベースに、機械学習による土地被覆分類に進む。

---
## Exercise Log の提出

以下の内容を Exercise Log（テンプレートは `exercises/template.md`）に記入し、
指定の提出方法（GitHub push または Colab + Discord）で提出すること。

**必須:**
- [ ] 課題①（Step 1-3）選択した条件番号・観測日・軌道方向・偏波の記録
- [ ] 課題①（Step 3-4）SARと光学の比較 — 「光学では区別できないがSARでは異なる」場所の特定と理由
- [ ] 課題②：AIの出力内容と、物理的誤りの指摘・正しい説明

**発展（いずれか1つ以上）:**
- [ ] C1: 時期の比較（5月 vs 7月）と考察
- [ ] C2: VV-VH差マップの観察結果
- [ ] C3: 時系列グラフと作付けサイクルの考察
